In [28]:
# ==========================================
# MODULE 4: MODELING & PREDICTION
# Linear Regression (Stock Returns Model)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------------------------------
# 1️⃣ LOAD DATA
# ------------------------------------------

df = pd.read_csv("../data/processed/features_dataset.csv", low_memory=False)
df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"])
print([col for col in df.columns if "return" in col])
list(df)



['returnOnAssets', 'returnOnEquity', 'return_1d', 'return_5d', 'return_10d', 'return_21d', 'risk_adjusted_return']


/var/folders/kr/8p0vbgj13wvcz3hlcpdwqkh40000gn/T/ipykernel_94399/1662433961.py:17: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["Date"] = pd.to_datetime(df["Date"])


['Ticker',
 'Date',
 'Open',
 'High',
 'Low',
 'Close',
 'Volume',
 'Dividends',
 'Stock Splits',
 'Year',
 'address1',
 'city',
 'state',
 'zip',
 'country',
 'phone',
 'website',
 'industry',
 'industryKey',
 'industryDisp',
 'sector',
 'sectorKey',
 'sectorDisp',
 'longBusinessSummary',
 'fullTimeEmployees',
 'companyOfficers',
 'auditRisk',
 'boardRisk',
 'compensationRisk',
 'shareHolderRightsRisk',
 'overallRisk',
 'governanceEpochDate',
 'compensationAsOfEpochDate',
 'irWebsite',
 'executiveTeam',
 'maxAge',
 'priceHint',
 'previousClose',
 'open',
 'dayLow',
 'dayHigh',
 'regularMarketPreviousClose',
 'regularMarketOpen',
 'regularMarketDayLow',
 'regularMarketDayHigh',
 'dividendRate',
 'dividendYield',
 'exDividendDate',
 'payoutRatio',
 'fiveYearAvgDividendYield',
 'beta',
 'trailingPE',
 'forwardPE',
 'volume',
 'regularMarketVolume',
 'averageVolume',
 'averageVolume10days',
 'averageDailyVolume10Day',
 'bid',
 'ask',
 'bidSize',
 'askSize',
 'marketCap',
 'nonDilutedMarke

In [30]:
# ------------------------------------------
# 2️⃣ DEFINE TARGET (IMPORTANT)
# ------------------------------------------
# You ALREADY HAVE returns in dataset → use one of them

target_col = "return_1d"   # BEST choice for baseline model

# Optional alternatives:
# target_col = "return_5d"
# target_col = "return_10d"

# ------------------------------------------
# 3️⃣ DEFINE FEATURES (IMPORTANT)
# ------------------------------------------

feature_cols = [

    # Price & trend
    "Close",
    "Open",
    "High",
    "Low",

    # Moving averages
    "ma_5",
    "ma_10",
    "ma_21",
    "ema_5",
    "ema_10",
    "ema_21",
    "ema_12",
    "ema_26",

    # Ratios (VERY IMPORTANT SIGNALS)
    "price_ma5_ratio",
    "price_ma10_ratio",
    "price_ma21_ratio",

    # Momentum
    "momentum_5",
    "momentum_10",
    "momentum_21",
    "momentum_5_pct",
    "momentum_10_pct",
    "momentum_21_pct",

    # Volatility
    "vol_5",
    "vol_10",
    "vol_21",
    "range_pct",

    # Volume signals
    "volume",
    "volume_avg_5",
    "volume_avg_10",
    "volume_avg_21",
    "volume_momentum_5",
    "volume_price_ratio",

    # Technical indicators
    "rsi_14",
    "macd",
    "macd_signal",

    # Risk
    "risk_adjusted_return",

    # Position in 52-week range
    "52w_position"
]

# ------------------------------------------
# 4️⃣ REMOVE MISSING TARGET ROWS
# ------------------------------------------

df = df.dropna(subset=[target_col])

# ------------------------------------------
# 5️⃣ TIME-BASED TRAIN/TEST SPLIT
# ------------------------------------------

split_date = df["Date"].quantile(0.8)

train_df = df[df["Date"] <= split_date]
test_df  = df[df["Date"] > split_date]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test  = test_df[feature_cols]
y_test  = test_df[target_col]

# ------------------------------------------
# 6️⃣ TRAIN MODEL
# ------------------------------------------

model = LinearRegression()
model.fit(X_train, y_train)

# ------------------------------------------
# 7️⃣ PREDICT
# ------------------------------------------

y_pred = model.predict(X_test)

# ------------------------------------------
# 8️⃣ EVALUATE MODEL
# ------------------------------------------

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("\n📊 MODEL PERFORMANCE")
print("RMSE:", rmse)
print("MAE :", mae)
print("R²  :", r2)

# ------------------------------------------
# 9️⃣ FEATURE IMPORTANCE (VERY IMPORTANT FOR RECRUITERS)
# ------------------------------------------

coefficients = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", ascending=False)

print("\n📈 TOP POSITIVE DRIVERS:")
print(coefficients.head(10))

print("\n📉 TOP NEGATIVE DRIVERS:")
print(coefficients.tail(10))

print("\nIntercept:", model.intercept_)

# ------------------------------------------
# 🔟 STORE PREDICTIONS
# ------------------------------------------

test_df = test_df.copy()
test_df["Predicted_Return"] = y_pred

# Trading signal
test_df["Signal"] = (test_df["Predicted_Return"] > 0).astype(int)

# ------------------------------------------
# 1️⃣1️⃣ SIMPLE BACKTEST
# ------------------------------------------

test_df["Strategy_Return"] = test_df["Signal"] * test_df[target_col]

test_df["Cumulative_Market"] = (1 + test_df[target_col]).cumprod()
test_df["Cumulative_Strategy"] = (1 + test_df["Strategy_Return"]).cumprod()

print("\n📊 STRATEGY VS MARKET")
print(test_df[["Cumulative_Market", "Cumulative_Strategy"]].tail())

# ------------------------------------------
# SAVE OUTPUT
# ------------------------------------------

test_df.to_csv("../data/processed/linear_model_predictions.csv", index=False)

print("\n✅ DONE: Model trained and predictions saved.")


📊 MODEL PERFORMANCE
RMSE: 0.013269285761108054
MAE : 0.00770975641767314
R²  : 0.6611142064204152

📈 TOP POSITIVE DRIVERS:
             Feature  Coefficient
10            ema_12     1.761548
32              macd     1.332421
11            ema_26     0.429127
12   price_ma5_ratio     0.067330
22            vol_10     0.057046
13  price_ma10_ratio     0.042020
18    momentum_5_pct     0.041150
19   momentum_10_pct     0.015266
0              Close     0.007757
21             vol_5     0.004320

📉 TOP NEGATIVE DRIVERS:
             Feature  Coefficient
3                Low    -0.000697
14  price_ma21_ratio    -0.000820
1               Open    -0.000926
20   momentum_21_pct    -0.001252
23            vol_21    -0.073764
7              ema_5    -0.074798
9             ema_21    -0.086004
24         range_pct    -0.117680
33       macd_signal    -0.634792
8             ema_10    -2.035946

Intercept: -0.1038122158591441

📊 STRATEGY VS MARKET
       Cumulative_Market  Cumulative_Strategy
125

In [8]:
list(df)

['Date',
 'Open',
 'High',
 'Low',
 'Close',
 'Volume',
 'Dividends',
 'Stock Splits',
 'Year',
 'address1',
 'city',
 'state',
 'zip',
 'country',
 'phone',
 'website',
 'industry',
 'industryKey',
 'industryDisp',
 'sector',
 'sectorKey',
 'sectorDisp',
 'longBusinessSummary',
 'fullTimeEmployees',
 'companyOfficers',
 'auditRisk',
 'boardRisk',
 'compensationRisk',
 'shareHolderRightsRisk',
 'overallRisk',
 'governanceEpochDate',
 'compensationAsOfEpochDate',
 'irWebsite',
 'executiveTeam',
 'maxAge',
 'priceHint',
 'previousClose',
 'open',
 'dayLow',
 'dayHigh',
 'regularMarketPreviousClose',
 'regularMarketOpen',
 'regularMarketDayLow',
 'regularMarketDayHigh',
 'dividendRate',
 'dividendYield',
 'exDividendDate',
 'payoutRatio',
 'fiveYearAvgDividendYield',
 'beta',
 'trailingPE',
 'forwardPE',
 'volume',
 'regularMarketVolume',
 'averageVolume',
 'averageVolume10days',
 'averageDailyVolume10Day',
 'bid',
 'ask',
 'bidSize',
 'askSize',
 'marketCap',
 'nonDilutedMarketCap',
 'fi